In [1]:
import cdms
import numpy as np
import os, sys 
import ROOT
from cats.cdataframe import CDataFrame
from CDMSDataCatalog import CDMSDataCatalog
import supercuts
import glob
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline

#CDMS = '/project/6049244/perry/CDMS/' # set in .bash_profile
#stylesheet = os.path.join(CDMS,"scripts","stylesheets","blues.mplstyle")
#plt.style.use(stylesheet)

#sys.path.append(os.path.join(os.path.join(CDMS,"scripts")))
#import setup

/usr/local/lib/python3.10/dist-packages/CDMSDataCatalog/CDMSDataCatalog.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Welcome to JupyROOT 6.28/10


In [2]:
from scipy.optimize import curve_fit
from uncertainties import *
from uncertainties.umath import exp
import math

def func(x, a, b):
    return a*x + b

def fit_logPulse(x, y, start, stop):
    popt, pcov = curve_fit(func, x[start:stop], y[start:stop])
    return popt[0], np.sqrt(np.diag(pcov))[0], popt[1], np.sqrt(np.diag(pcov))[1]

def str_with_err(value, error):
    digits = -int(math.floor(math.log10(error)))
    return "{0:.{2}f}({1:.0f})".format(value, error*10**digits, digits)

### Propagate error for Geant4 SourceSim result

Found that simulating 1E9 Cf-252 decays in the CUTE fridge results in 9208 Ge-70 + n $\to$ Ge-71 interactions.  
Conservative uncertainty for Geant4 hadronic physics is 4%.  
Poisson uncertainty for total count.

In [3]:
def tot_err(count):
    poisson_err = np.sqrt(count)
    geant4_err = count * 0.04
    pos_err = count * 0.076

    err = np.sqrt(poisson_err ** 2 + geant4_err ** 2 + pos_err ** 2)
    decays = ufloat(count, err)
    
    return decays

In [4]:
print('Number of K shell decays', end=": ")
K_decays = tot_err(int(8059))
print('{:+.3uS}'.format(K_decays))

print('Number of L shell decays', end=": ")
L_decays = tot_err(int(959))
print('{:+.3uS}'.format(L_decays))

print('Number of M shell decays', end=": ")
M_decays = tot_err(int(177))
print('{:+.3uS}'.format(M_decays))

print('Total number of decays', end=": ")
total_decays = tot_err(int(9195))
print('{:+.3uS}'.format(total_decays))

Number of K shell decays: +8059(698)
Number of L shell decays: +959.0(88.0)
Number of M shell decays: +177.0(20.2)
Total number of decays: +9195(796)


In [5]:
K_over_tot = K_decays / total_decays * 100
print('{:+.2uS}'.format(K_over_tot))

L_over_tot = L_decays / total_decays * 100
print('{:+.2uS}'.format(L_over_tot))

M_over_tot = M_decays / total_decays * 100
print('{:+.2uS}'.format(M_over_tot))

+88(11)
+10.4(1.3)
+1.92(28)


### Propagate errors in estimation of number of decays

[Cf-252 half-life](https://www.chemlin.org/isotope/californium-252)  
[Ge-71 half-life](https://www.chemlin.org/isotope/germanium-71)

In [6]:
# Z1 0V R37
#series_list=['23240104_051944', '23231222_074513', 
#'23231221_235414', '23231221_223301', 
#'23231221_162803']
# Z1 0V R38
series_list=['23240310_103543', '23240310_045613',
'23240310_015547', '23240305_084620', 
'23240305_050626', '23240305_030542', 
'23240221_012437']

In [7]:
ProdTag = 'CUTE_T3GeCalib_NxM_P4.0.0_V05-06_C0.3.6'

In [8]:
filepath = [f'/scratch/perry/CDMS/CUTE/R38/Processed/Releases/{ProdTag}/Submerged/{ProdTag}_{i}.root' for i in series_list]

In [9]:
det = 1 # detector number

df = CDataFrame("rqDir/zip"+str(det), filepath, friends = [[x+":rqDir/eventTree" for x in filepath]])

In [10]:
## Apply some basic data quality filters and get the RQs you're interested in
logfile = '"cute_tower3testing.log"'
df = df.Define("LEDLogFile", logfile) 
df = df.CDefine("LEDOn", supercuts.ledOn_old, ["EventTime", "LEDLogFile"])
df = df.Filter("!LEDOn")
df_filtered = df.Filters(["TriggerType == 1", "TriggerDetectorNum=="+str(det), "PTOFamps>0"])

In [11]:
RQs = (["SeriesNumber", "PTOFamps", "EventNumber", "EventTime"])
df_rqs = df_filtered.AsNumpy(RQs)

In [12]:
import datetime
from zoneinfo import ZoneInfo

In [13]:
y2s = 3.156e7
d2s = 86400
m2s = 60

# Cf252 radioactive properties
Cf252_halflife = ufloat(2.645 * y2s, 0.008 * y2s) # s
Cf252_lifetime = Cf252_halflife / np.log(2) # s
Cf252_lambda   = 1 / Cf252_lifetime # Hz

# Ge-71 radioactive properties
Ge71_halflife = ufloat(11.468 * d2s, 0.008 * d2s) # s
Ge71_lifetime = Ge71_halflife / np.log(2) # s
Ge71_lambda   = 1 / Ge71_lifetime

In [14]:
Cf252_activity = ufloat(37000, 37000 * 0.15)                          # activity of Cf252 at reference time; given in Hz
t_ref          = datetime.datetime(2020, 3, 15, 00, 0, tzinfo=ZoneInfo("America/New_York")).timestamp()    # reference time of nominal activity; Unix time
activ_prob     = total_decays * 1e-9                                  # Ge activations per neutron emitted; found using sourcesim w/ CUTE geometry

# rate of Ge activation during exposure
A = Cf252_activity * activ_prob # activations / second

In [41]:
start_times = pd.read_csv('../source_exposure_start_times.csv')
end_times = pd.read_csv('../source_exposure_end_times.csv')

timestamp_err = 30 * m2s

exposures = {i: {} for i in range(9)}

for i in range(9):
    year_0, year_f   = start_times['year'][i], end_times['year'][i]
    month_0, month_f = start_times['month'][i], end_times['month'][i]
    day_0, day_f     = start_times['day'][i], end_times['day'][i]
    hour_0, hour_f   = start_times['hour'][i], end_times['hour'][i]
    min_0, min_f     = start_times['minute'][i], end_times['minute'][i]
    s_0, s_f         = start_times['second'][i], end_times['second'][i]
    if i < 8:
        exposures[i]['t0'] = ufloat(datetime.datetime(year_0, month_0, day_0, hour_0, min_0, s_0, tzinfo=ZoneInfo("America/New_York")).timestamp(), timestamp_err)
        exposures[i]['tf'] = ufloat(datetime.datetime(year_f, month_f, day_f, hour_f, min_f, s_f, tzinfo=ZoneInfo("America/New_York")).timestamp(), timestamp_err)
        exposures[i]['dt'] = exposures[i]['tf'] - exposures[i]['t0']
    else:
        exposures[i]['t0'] = ufloat(datetime.datetime(year_0, month_0, day_0, hour_0, min_0, s_0, tzinfo=ZoneInfo("America/New_York")).timestamp(), d2s)
        exposures[i]['tf'] = ufloat(datetime.datetime(year_f, month_f, day_f, hour_f, min_f, s_f, tzinfo=ZoneInfo("America/New_York")).timestamp(), d2s)
        exposures[i]['dt'] = exposures[i]['tf'] - exposures[i]['t0']

In [42]:
# find total decays with N = int[ A exp( -lambda_Cf252 (t - tref) ) ] dt
# t from t0 to T
def integrate_decays(T, t0):
    return (Cf252_activity * (-exp(Cf252_lambda * (t_ref - T)) + exp(Cf252_lambda * (t_ref - t0)))) / Cf252_lambda

In [43]:
#### starting with 0 decays ####
N_tot = 0

#### track change in number of Cf-252 decays during exposures ####
for i in range(len(exposures)):
    N = integrate_decays(exposures[i]['tf'], exposures[i]['t0'])

    N_tot += N

print('{:+.3uS} total Cf-252 decays'.format(N_tot))

+2.211(369)e+10 total Cf-252 decays


In [44]:
# find total activations with N = int[ A exp( -lambda_Cf252 (t + t0) ) * exp(-lambda_Ge71 (T - t) ) ] dt + N0 exp(-lambda_Ge71 T) 
# t from t0 to T
def integrate_activations(N, T, t0):
    return (A / (Cf252_lambda - Ge71_lambda) * ( exp(- Cf252_lambda * t0 - Ge71_lambda * T) - exp(-Cf252_lambda * (T + t0)) ) + 
            N * exp(-Ge71_lambda * T) )

In [45]:
#### starting with 0 nuclei activated ####
N = 0

#### track change in number of Ge-71 nuclei during and after exposures ####
for i in range(len(exposures)):
    N = integrate_activations(N, exposures[i]['dt'], exposures[i]['t0'] - t_ref)
    if (i != len(exposures) - 1):
        N = N * exp(-Ge71_lambda * (exposures[i+1]['t0'] - exposures[i]['tf']))
    else:
        # check change in number of Ge-71 during each series
        N_tot = 0
        for sn in np.unique(df_rqs['SeriesNumber']):
            snCut = df_rqs['SeriesNumber'] == sn
            evtTimes = df_rqs['EventTime'][snCut]
            sn_start = min(evtTimes)
            sn_end = max(evtTimes)

            N_start = N * exp(-Ge71_lambda * (sn_start - (exposures[len(exposures) - 1]['t0'] + exposures[len(exposures) - 1]['dt'])))
            N_end = N * exp(-Ge71_lambda * (sn_end - (exposures[len(exposures) - 1]['t0'] + exposures[len(exposures) - 1]['dt'])))
            #print(f'{np.round(N_start - N_end)} Ge-71 events in series {int(sn)}')

            N_tot += N_start - N_end
            print(f'{sn}: {N_start - N_end}')

print()
print('{:+.3uS} total Ge-71 events'.format(N_tot))
print('{:+.3uS} K shell events'.format(N_tot * K_over_tot / 100))
print('{:+.3uS} L shell events'.format(N_tot * L_over_tot / 100))
print('{:+.3uS} M shell events'.format(N_tot * M_over_tot / 100))

23240221012437.0: (4.8+/-1.6)e+02
23240305030542.0: 62+/-21
23240305050626.0: (1.1+/-0.4)e+02
23240305084620.0: 61+/-20
23240310015547.0: 46+/-15
23240310045613.0: (1.3+/-0.4)e+02
23240310103543.0: 72+/-24

+962(320) total Ge-71 events
+843(281) K shell events
+100.3(33.5) L shell events
+18.52(6.32) M shell events
